# Smoothing Filters: EMA vs Moving Average vs Kalman

An interactive comparison of three classic smoothing filters applied to a noisy altitude signal from a model rocket (boost + ballistic coast).

Use the sliders below to change the noise level and filter parameters and watch how each filter responds.

- **SMA** — Simple Moving Average (sliding window mean).
- **EMA** — Exponential Moving Average (single-pole IIR low-pass).
- **Kalman** — constant-velocity Kalman filter.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Checkbox

%matplotlib inline

# Model rocket altitude: powered boost + ballistic coast
g = 9.81
a_boost = 80          # boost acceleration excluding gravity (m/s^2)
t_burn  = 0.8         # motor burn time (s)

t = np.arange(0, 15, 0.02)   # 50 Hz sampling
dt = t[1] - t[0]

boost = t <= t_burn
true_sig = np.empty_like(t)
true_sig[boost] = 0.5 * (a_boost - g) * t[boost]**2

h_burn = 0.5 * (a_boost - g) * t_burn**2
v_burn = (a_boost - g) * t_burn
tc = t[~boost] - t_burn
true_sig[~boost] = h_burn + v_burn*tc - 0.5*g*tc**2
true_sig = np.maximum(true_sig, 0)

landed = np.where((true_sig == 0) & (t > t_burn))[0]
if len(landed):
    end = landed[0] + 1
    t = t[:end]; true_sig = true_sig[:end]

rng = np.random.default_rng(0)
unit_noise = rng.standard_normal(t.size)

## Filter implementations

In [ ]:
def ema(x, alpha):
    y = np.empty_like(x)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = alpha*x[i] + (1-alpha)*y[i-1]
    return y

def sma(x, window):
    padded = np.concatenate([np.full(window-1, x[0]), x])
    return np.convolve(padded, np.ones(window)/window, mode='valid')

def kalman(x, dt, q, r):
    F = np.array([[1, dt], [0, 1]])
    H = np.array([[1, 0]])
    Q = np.array([[dt**4/4, dt**3/2],
                  [dt**3/2, dt**2]]) * q
    R = np.array([[r]])
    I = np.eye(2)
    state = np.array([x[0], 0.0])
    P = np.eye(2)
    out = np.empty_like(x)
    for i in range(len(x)):
        state = F @ state
        P = F @ P @ F.T + Q
        y_innov = x[i] - (H @ state)[0]
        S = (H @ P @ H.T + R)[0, 0]
        K = (P @ H.T).flatten() / S
        state = state + K * y_innov
        P = (I - np.outer(K, H)) @ P
        out[i] = state[0]
    return out

## Interactive comparison

Move the sliders to change the measurement noise and each filter's parameters.

In [ ]:
@interact(
    noise=FloatSlider(min=0.0, max=10.0, step=0.1, value=2.0,
                      description='Noise σ (m)', continuous_update=False),
    show_ema=Checkbox(value=True,  description='Show EMA'),
    alpha=FloatSlider(min=0.01, max=1.0, step=0.01, value=0.10,
                     description='EMA α', continuous_update=False),
    show_sma=Checkbox(value=True,  description='Show SMA'),
    window=IntSlider(min=1, max=100, step=1, value=20,
                     description='SMA window', continuous_update=False),
    show_kalman=Checkbox(value=True,  description='Show Kalman'),
    q=FloatLogSlider(value=0.5, base=10, min=-5, max=5, step=0.1,
                     description='Kalman q', continuous_update=False),
)
def plot_filters(noise, show_ema, alpha, show_sma, window, show_kalman, q):
    noisy = true_sig + noise * unit_noise
    r = max(noise, 0.01) ** 2

    plt.figure(figsize=(11, 5))
    plt.plot(t, noisy,    color='lightgray', lw=0.8, label='Noisy')
    plt.plot(t, true_sig, 'k--', lw=1.2, label='True')

    if show_sma:
        plt.plot(t, sma(noisy, window),       color='royalblue', lw=2, label=f'SMA window={window}')
    if show_ema:
        plt.plot(t, ema(noisy, alpha),        color='crimson',   lw=2, label=f'EMA α={alpha:.2f}')
    if show_kalman:
        plt.plot(t, kalman(noisy, dt, q, r),  color='seagreen',  lw=2, label=f'Kalman q={q:.3g}')

    plt.xlabel('Time (s)')
    plt.ylabel('Altitude (m)')
    plt.title(f'EMA vs Moving Average vs Kalman Filter  (σ = {noise:.1f} m)')
    plt.legend(loc='upper right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()